# 💻 Laboratório Prático - Aula 14: Engenharia de Software para Dados

**Objetivo:** Consolidar a divisão segura de dados (Treino/Teste), entender a estrutura de Classes em Python e exportar lógicas de transformação customizadas para arquivos limpos (`.py`), garantindo o isolamento perfeito num *Pipeline*.

---

## 🔥 Aquecimento 1: A Divisão 80/20 (O Aluno e a Prova)
Antes de construir nossa "peça de Lego", precisamos separar o material de estudo (Treino) da prova final (Teste). Usaremos a função `train_test_split` do Scikit-Learn.

**Sua Tarefa:** Execute o código abaixo para gerar nossos dados simulados e preencha as lacunas para dividi-los destinando 20% para teste.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Gerando dados simulados de transações
np.random.seed(42)
dados = pd.DataFrame({
    'Data_Transacao': pd.date_range(start='2025-01-01', periods=100, freq='D'),
    'Valor': np.random.normal(150, 50, 100),
    'Target_Fraude': np.random.choice([0, 1], 100, p=[0.9, 0.1])
})

# Separando Preditores (X) e Alvo (y)
X = dados.drop('Target_Fraude', axis=1)
y = dados['Target_Fraude']

# ESCREVA SEU CÓDIGO AQUI: 
# Separe X e y em treino (80%) e teste (20%)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamanho do Treino (Material de Estudo): {X_treino.shape} linhas")
print(f"Tamanho do Teste (Prova Final): {X_teste.shape} linhas")


Tamanho do Treino (Material de Estudo): (80, 2) linhas
Tamanho do Teste (Prova Final): (20, 2) linhas


---

## 🔥 Aquecimento 2: O Molde do Lego (POO Básica)
O Scikit-Learn exige que nossas peças customizadas obedeçam a um contrato: elas devem herdar as classes `BaseEstimator` e `TransformerMixin`. Isso nos dá a integração automática com Pipelines.

**Sua Tarefa:** Preencha os campos vazios para montar o esqueleto da classe `ConversorTemporal` e suas regras.

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin

# ESCREVA SEU CÓDIGO AQUI:
class ConversorTemporal(BaseEstimator, TransformerMixin):
    
    # O método construtor (Botão de ligar)
    def __init__(self):
        pass
    
    # A Fase de Estudo (Enxerga só os 80%)
    def fit(self, X, y=None):
        # Como não precisamos aprender estatísticas do passado para manipular datas,
        # apenas retornamos o próprio objeto (self)
        return self
    
    # A Fase de Prova (Aplica a transformação)
    def transform(self, X):
        # 1. Boa prática: Criar uma cópia para não alterar o dado original por acidente
        X_modificado = X.copy()
        
        # 2. Injetando a regra de negócio: Converter string para datetime e checar fim de semana
        X_modificado['Data_Transacao'] = pd.to_datetime(X_modificado['Data_Transacao'])
        
        # Cria a flag 'Is_Weekend' (>= 5 significa Sábado ou Domingo)
        X_modificado['Is_Weekend'] = (X_modificado['Data_Transacao'].dt.dayofweek >= 5).astype(int)
        
        # 3. Descarta a data original para não confundir o modelo
        X_modificado = X_modificado.drop('Data_Transacao', axis=1)
        
        return X_modificado

print("Sintaxe da classe completada com sucesso!")


Sintaxe da classe completada com sucesso!


---

## 🛠️ Laboratório (Exportação)
> **Início do laboratório prático. Os alunos deverão extrair o código dessa classe complexa de dentro do Jupyter Notebook e salvá-lo em um arquivo em branco chamado `custom_transformers.py` usando o VS Code ou editor de texto padrão.**

**Sua Tarefa:**
1. Abra o seu **VS Code** (ou editor de preferência) na mesma pasta onde este Notebook está salvo.
2. Crie um arquivo chamado **`custom_transformers.py`**.
3. Copie o código da classe `ConversorTemporal` que você completou no Aquecimento 2 e cole neste arquivo.
4. **⚠️ ATENÇÃO E IMPORTAÇÃO VITAL:** Arquivos `.py` são universos fechados (não enxergam o que você rodou no Jupyter). Portanto, você é **obrigado** a colocar as seguintes linhas na primeira linha do seu arquivo `custom_transformers.py` antes de colar o resto do código:

```python
# CONTEÚDO EXATO QUE DEVE ESTAR NO TOPO DO SEU ARQUIVO custom_transformers.py:

import pandas as pd # <-- Sem isso, a conversão de datas vai falhar!
from sklearn.base import BaseEstimator, TransformerMixin

# -- AGORA COLE ABAIXO A SUA CLASSE --
```


---

## 🚀 Teste e Importação
> **Voltar ao Jupyter Notebook. Rodar `from custom_transformers import ConversorTemporal`. Dividir a base real usando `train_test_split` (80/20), colocar o objeto recém-criado dentro de um Pipeline e testar se ele executa o fluxo inteiro sem vazar dados do futuro para o passado.**

**Sua Tarefa Final:** Preencha as lacunas e execute o código abaixo. Vamos plugar a nossa "peça de Lego caseira" com uma peça oficial do Scikit-Learn (o `StandardScaler`) e garantir o isolamento estatístico!

In [5]:
# 1. Importar a nossa classe diretamente do arquivo recém-criado
from custom_transformers import ConversorTemporal 

# 2. Importar o ecossistema de Pipelines e Scalers do Scikit-Learn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 3. Construindo a Linha de Montagem (Pipeline)
# Instancie o conversor temporal e o escalonador
fluxo_preparacao = Pipeline(steps=[
    ('extrator_datas', ConversorTemporal()),  # Nossa peça caseira conectada
    ('escalonador', StandardScaler())         # Peça oficial do Sklearn conectada
])

# 4. A Prova Final: Rodando o fluxo SEM vazar dados
# O método que ESTUDA e APLICA de uma vez, apenas nos 80% (Treino)
X_treino_pronto = fluxo_preparacao.fit_transform(X_treino)

# O método que faz a prova final (apenas APLICA) nos 20% (Teste) sem recalcular métricas!
X_teste_pronto = fluxo_preparacao.transform(X_teste)

print("✅ Pipeline executado com sucesso!")
print(f"Formato final do Treino processado: {X_treino_pronto.shape}")
print(f"Formato final do Teste processado: {X_teste_pronto.shape}")

# Verificando as primeiras linhas para confirmar a presença da coluna 'Is_Weekend' tratada e escalonada
print("\nAmostra dos dados de Treino (já convertidos e escalonados):")
print(X_treino_pronto[:3])


✅ Pipeline executado com sucesso!
Formato final do Treino processado: (80, 2)
Formato final do Teste processado: (20, 2)

Amostra dos dados de Treino (já convertidos e escalonados):
[[ 1.1286706  -0.55809982]
 [-0.44251218  1.79179416]
 [-1.11057805 -0.55809982]]
